In [1]:
import numpy as np
import myutils.tge.grid as gd
import myutils.clfuncs.clfuncs as cfunc
import myutils.scf.scf as sf

Imported myutils
Imported Grid,      Import Time : 01/06/26 | 11:14:56 AM
Imported clfuncs,   Import Time : 01/06/26 | 11:14:56 AM
Imported SCF,       Import Time : 01/06/26 | 11:14:56 AM


In [2]:
# load bining info
BIN = np.load('/home/cts23ph/git_package_myutils/tge/bin_info_1000007200.npz')

# Set the gridding parameters in the TGE code
n1      = 0                  # Starting channel number
n2      = 767                # End channel number
Flag    = True               # Apply actual flagging of data
Umax    = int(BIN['Umax'])   # Baselines greater than this are rejected
FWHM    = int(BIN['FWHM'])   # Primary beam FWHM of the telescope in degrees
f       = float(BIN['f'])    # Tapering parameter value
nstokes = [0,1]              # 0 for XX and 1 for YY (polarizations to grid)
# ===================================================
 
mask = BIN['NI'] >= 0        # mask array
Nbin = int(BIN['Nbin'])      # No of annular bins the whole uv plane has been divided into
ni   = BIN['ni']             # Contains info about which grid point fall into which bin
lval = BIN['lval']           # obtain the ell values

In [3]:
infile        = '/home/MWA_data/1000007200.fits'                                            # fits file name
out_uaps_fits = '/home/cts23ph/git_package_myutils/simvis/simulated/1000007200_uaps.fits'   # uaps fits file name

In [4]:
import os
index = int(os.path.split(infile)[1].split(".")[0][-4:])
print(index)

7200


In [5]:
# scf parameters
SM = 2 # smmothing scale in MHz
NW  = int(SM/0.04)   # N # = 50 if SM = 2 MHz and dnuc = 0.04 MHz

In [6]:
# nc = n2-n1+1-2*NW number of available channels

### $\textrm{Apply to Data to get}\,\,\,\quad e_{\ell}(\nu_a,\nu_b)$

In [7]:
# grid the data first then correlate
nrel = -1 # for data
GV_data  = (gd.grid(infile, n1, n2, nrel, Umax, FWHM, f, Flag, nstokes)[0])[:, mask]

# perform SCF
GVr_data = sf.doscf(GV_data, SM, window = 'hann', method = 'fft')

# correlate 
el      = cfunc.correlate(GV_data[..., NW:-NW], ni, Nbin) # without scf
elr     = cfunc.correlate(GVr_data            , ni, Nbin) # with    scf

# save el file
np.save(f'el_maps_{index}.npy' , el)
np.save(f'elr_maps_{index}.npy', elr)

<<<<<<<<<<< TGE >>>>>>>>>>>>
<< 01/06/26 | 11:14:56 AM >>
Type  : Data
<< 01/06/26 | 11:16:12 AM >>
Elapsed   :   00:01:15.86
<<<<<<<<<<< Done >>>>>>>>>>>
<============= Performing SCF =============>
<=== Start Time: 01/06/26 | 11:16:13 AM ===>
Convoluting kernel : fft
Window function    : Hanning
Smoothing scale    : 2 MHz
<=== End Time  : 01/06/26 | 11:16:14 AM ===>
<=== Elapsed   : 00:00:01.69, 1.685 seconds.
<================ Done SCF ================>
Channels   : 668
Time Taken : 4.694 seconds.

Channels   : 668
Time Taken : 4.208 seconds.



In [8]:
print(el.shape)  # (Nbin, nc = n2-n1+1, nc)
print(elr.shape) # (Nbin, nc = n2-n1+1, nc)

(20, 668, 668)
(20, 668, 668)


### $\textrm{Apply to UAPS to get}\quad m_{\ell}(\nu_a,\nu_b)$

In [9]:
ml  = np.zeros((100, Nbin, (n2-n1+1-2*NW), (n2-n1+1-2*NW))) 

for reln in range(ml.shape[0]):
    nrel = reln   # for uaps
    print(f'\n<<< Reln = {nrel} >>>\n')
    GV_uaps  = (gd.grid(out_uaps_fits, n1, n2, nrel, Umax, FWHM, f, Flag, nstokes)[0])[:, mask, NW:-NW]
    
    ml[reln] = cfunc.correlate(GV_uaps, ni, Nbin)

ml = np.mean(ml, axis = 0) # ensemble average

# save ml file
np.save(f'ml_maps_{index}.npy', ml)


<<< Reln = 0 >>>

<<<<<<<<<<< TGE >>>>>>>>>>>>
<< 01/06/26 | 11:16:23 AM >>
Type  : UAPS
Copy  : 0 channel.
<< 01/06/26 | 11:17:36 AM >>
Elapsed   :   00:01:13.23
<<<<<<<<<<< Done >>>>>>>>>>>
Channels   : 668
Time Taken : 4.084 seconds.


<<< Reln = 1 >>>

<<<<<<<<<<< TGE >>>>>>>>>>>>
<< 01/06/26 | 11:17:41 AM >>
Type  : UAPS
Copy  : 1 channel.
<< 01/06/26 | 11:18:54 AM >>
Elapsed   :   00:01:13.33
<<<<<<<<<<< Done >>>>>>>>>>>
Channels   : 668
Time Taken : 4.074 seconds.


<<< Reln = 2 >>>

<<<<<<<<<<< TGE >>>>>>>>>>>>
<< 01/06/26 | 11:18:58 AM >>
Type  : UAPS
Copy  : 2 channel.
<< 01/06/26 | 11:20:11 AM >>
Elapsed   :   00:01:13.08
<<<<<<<<<<< Done >>>>>>>>>>>
Channels   : 668
Time Taken : 4.071 seconds.


<<< Reln = 3 >>>

<<<<<<<<<<< TGE >>>>>>>>>>>>
<< 01/06/26 | 11:20:16 AM >>
Type  : UAPS
Copy  : 3 channel.
<< 01/06/26 | 11:21:29 AM >>
Elapsed   :   00:01:13.13
<<<<<<<<<<< Done >>>>>>>>>>>
Channels   : 668
Time Taken : 4.074 seconds.


<<< Reln = 4 >>>

<<<<<<<<<<< TGE >>>>>>>>>

In [10]:
print(ml.shape) # (Nbin, nc = n2-n1+1, nc)

(20, 668, 668)


### $\textrm{Apply to Noise}\quad n_{\ell}(\nu_a,\nu_b)$

In [11]:
nl  = np.zeros((50, Nbin, (n2-n1+1-2*NW), (n2-n1+1-2*NW))) # without scf
nlr = np.zeros((50, Nbin, (n2-n1+1-2*NW), (n2-n1+1-2*NW))) # with    scf

for iter_seed in range(nl.shape[0]):
    seedn = int(index+iter_seed)   # for noise
    
    print(f'\n<<< Noise = {seedn} >>>\n')
    nrel = -2       # noise
    GV_noise       = (gd.grid(infile, n1, n2, nrel, Umax, FWHM, f, Flag, nstokes, seedn)[0])[:, mask]

    # perform scf
    GVr_noise = sf.doscf(GV_noise, SM, window = 'hann', method = 'fft')
    
    # correlate 
    nl[iter_seed]      = cfunc.correlate(GV_noise[..., NW:-NW], ni, Nbin) # without scf
    nlr[iter_seed]     = cfunc.correlate(GVr_noise            , ni, Nbin) # with    scf
    
# save nl file
np.save(f'nl_maps_{index}.npy' , nl)
np.save(f'nlr_maps_{index}.npy', nlr)


<<< Noise = 7200 >>>

<<<<<<<<<<< TGE >>>>>>>>>>>>
<< 01/06/26 | 01:25:36 PM >>
Seed  : 7200
Type  : Noise only
<< 01/06/26 | 01:26:52 PM >>
Elapsed   :   00:01:15.79
<<<<<<<<<<< Done >>>>>>>>>>>
<============= Performing SCF =============>
<=== Start Time: 01/06/26 | 01:26:52 PM ===>
Convoluting kernel : fft
Window function    : Hanning
Smoothing scale    : 2 MHz
<=== End Time  : 01/06/26 | 01:26:54 PM ===>
<=== Elapsed   : 00:00:01.67, 1.675 seconds.
<================ Done SCF ================>
Channels   : 668
Time Taken : 4.090 seconds.

Channels   : 668
Time Taken : 4.134 seconds.


<<< Noise = 7201 >>>

<<<<<<<<<<< TGE >>>>>>>>>>>>
<< 01/06/26 | 01:27:02 PM >>
Seed  : 7201
Type  : Noise only
<< 01/06/26 | 01:28:18 PM >>
Elapsed   :   00:01:15.79
<<<<<<<<<<< Done >>>>>>>>>>>
<============= Performing SCF =============>
<=== Start Time: 01/06/26 | 01:28:18 PM ===>
Convoluting kernel : fft
Window function    : Hanning
Smoothing scale    : 2 MHz
<=== End Time  : 01/06/26 | 01:28:20 

In [12]:
print(nl.shape)  # (50, Nbin, nc = n2-n1+1, nc)
print(nlr.shape) # (50, Nbin, nc = n2-n1+1, nc)

(50, 20, 668, 668)
(50, 20, 668, 668)


### $\textrm{Calculate MAPS}\quad C_{\ell}(\nu_a,\nu_b) = e_{\ell}(\nu_a,\nu_b)/m_{\ell}(\nu_a,\nu_b)\longrightarrow C_{\ell}(\Delta\nu)$

In [13]:
# without scf
cl = el/ml
print(cl.shape)

cl_dnu = cfunc.cl_dnu_nua_nub(cl)
print(cl_dnu.shape)

# save cl_dnu file
np.save(f'cldnu_{index}.npy', cl_dnu)

(20, 668, 668)
(20, 668)


/tmp/ipykernel_918268/3749414482.py:2: RuntimeWarning: invalid value encountered in divide
  cl = el/ml


In [14]:
# with scf
clr = elr/ml
print(clr.shape)

cl_dnur = cfunc.cl_dnu_nua_nub(clr)
print(cl_dnur.shape)

# save cl_dnu file
np.save(f'cldnur_{index}.npy', cl_dnur)

(20, 668, 668)


/tmp/ipykernel_918268/682113152.py:2: RuntimeWarning: invalid value encountered in divide
  clr = elr/ml


(20, 668)


### $\textrm{Calculate MAPS for noise }\quad [C_{\ell}(\nu_a,\nu_b)]_{\rm noise} = n_{\ell}(\nu_a,\nu_b)/m_{\ell}(\nu_a,\nu_b)\longrightarrow [C_{\ell}(\Delta\nu)]_{\rm noise}$

In [15]:
# without scf
cl_noise = nl/ml
print(cl_noise.shape)

cl_dnu_noise = np.array([cfunc.cl_dnu_nua_nub(cl_noise[ii]) for ii in range(cl_noise.shape[0])])
print(cl_dnu_noise.shape)

# save cl_dnu file for noise
np.save(f'cldnu_noise_{index}.npy', cl_dnu_noise)

/tmp/ipykernel_918268/3792963694.py:2: RuntimeWarning: invalid value encountered in divide
  cl_noise = nl/ml


(50, 20, 668, 668)
(50, 20, 668)


In [16]:
# with scf
clr_noise = nlr/ml
print(clr_noise.shape)

cl_dnur_noise = np.array([cfunc.cl_dnu_nua_nub(clr_noise[ii]) for ii in range(clr_noise.shape[0])])
print(cl_dnur_noise.shape)

# save cl_dnu file for noise
np.save(f'cldnur_noise_{index}.npy', cl_dnur_noise)

/tmp/ipykernel_918268/390028080.py:2: RuntimeWarning: invalid value encountered in divide
  clr_noise = nlr/ml


(50, 20, 668, 668)
(50, 20, 668)
